# HTML panel test: a ripple tank computed on Kaggle

**Press Run All**, then **Start** in the panel under the last cell.
Cell 1 computes 121 runs of the 2D wave equation $u_{tt} = c^2(u_{xx}+u_{yy})$ (finite differences, leapfrog in time): 11 slit separations x 11 wavelengths, every position of the panel's two sliders, about 80 s on 4 cores. Cell 2 builds the panel in the Phyein Usul design: it only plays the stored frames. The same panel is saved as `ripple_tank.html` in the Output and opens in any browser.

In [ ]:
import numpy as np, json, time, base64, os, zlib
# ---- parameters (grid units: cell size 1, wave speed c = 1) ----
NX, NY      = 320, 200          # grid
DT          = 0.5               # time step (2D stability needs c*DT/dx < 0.707)
X_SRC       = 8                 # column that is driven (plane wave source)
X_WALL      = 70                # the wall with two slits (2 cells thick)
SLIT_W      = 6                 # slit width
SPONGE      = 30                # absorbing border thickness
X_SCREEN    = NX - SPONGE - 5   # where the bar chart samples the intensity
SEPARATIONS = list(range(20, 71, 5))   # slit centre-to-centre distance d: every slider position
WAVELENGTHS = list(range(10, 21))      # wavelength lambda: every slider position
STEPS       = 1400              # steps to reach the steady pattern
AVG_PERIODS = 4                 # periods averaged for the intensity bars
DOWN        = 2                 # stored frames are NX/2 x NY/2
FPP         = 12                # frames stored per wave period

y = np.arange(NY); x = np.arange(NX)
dist = np.minimum.reduce([np.maximum(0, SPONGE - (NX - 1 - x))[:, None] * np.ones((1, NY)),
                          np.ones((NX, 1)) * np.maximum(0, SPONGE - y)[None, :],
                          np.ones((NX, 1)) * np.maximum(0, SPONGE - (NY - 1 - y))[None, :]] , axis=0)
d_right = np.maximum(0, SPONGE - (NX - 1 - x))[:, None]; d_bot = np.maximum(0, SPONGE - y)[None, :]; d_top = np.maximum(0, SPONGE - (NY - 1 - y))[None, :]
d_left = np.maximum(0, X_SRC - x)[:, None] * 4
damp = np.exp(-0.002 * (np.maximum(np.maximum(d_right, d_left), np.maximum(d_bot, d_top))) ** 2)

def run(d, lam):
    wall = np.zeros((NX, NY), bool); wall[X_WALL:X_WALL + 2, :] = True
    for yc in (NY / 2 - d / 2, NY / 2 + d / 2):
        wall[X_WALL:X_WALL + 2, int(round(yc - SLIT_W / 2)):int(round(yc + SLIT_W / 2))] = False
    u = np.zeros((NX, NY)); up = np.zeros((NX, NY)); w = 2 * np.pi / lam; r = DT ** 2
    per = int(round(lam / DT)); frames = []; inten = np.zeros(NY); navg = 0
    keep = set(STEPS + np.round(np.arange(FPP) * per / FPP).astype(int))
    for n in range(STEPS + per):
        lap = -4 * u; lap[1:] += u[:-1]; lap[:-1] += u[1:]; lap[:, 1:] += u[:, :-1]; lap[:, :-1] += u[:, 1:]
        un = (2 * u - up + r * lap) * damp; un[wall] = 0
        t = (n + 1) * DT; un[X_SRC, :] = np.sin(w * t) * min(1.0, t / (3 * lam))
        up, u = u, un
        if n >= STEPS - AVG_PERIODS * per: inten += u[X_SCREEN] ** 2; navg += 1
        if n in keep: frames.append(u[::DOWN, ::DOWN].copy())
    F = np.array(frames); s = np.percentile(np.abs(F), 99.5)
    q = np.clip(np.round(F / s * 127 + 128), 0, 255).astype(np.uint8)
    I = inten / navg
    # far-field (Fraunhofer) two-slit formula on the same screen line, for comparison only (the screen is not far away)
    L = X_SCREEN - X_WALL; yy = y - NY / 2; sn = yy / np.sqrt(yy ** 2 + L ** 2)
    b = np.pi * SLIT_W * sn / lam
    fr = np.cos(np.pi * d * sn / lam) ** 2 * np.where(b == 0, 1, (np.sin(b) / np.where(b == 0, 1, b)) ** 2)
    return q, I, fr

RUNS = []; t0 = time.time()
for d in SEPARATIONS:
    for lam in WAVELENGTHS:
        t1 = time.time(); q, I, fr = run(d, lam)
        RUNS.append(dict(d=d, lam=lam, n=int(q.shape[0]), frames=base64.b64encode(zlib.compress(q.tobytes(), 9)).decode(),
                         I=np.round(I / I.max(), 4).tolist(), fr=np.round(fr, 4).tolist()))
        if len(RUNS) % 11 == 0: print(f"{len(RUNS)} of {len(SEPARATIONS) * len(WAVELENGTHS)} runs done; last d = {d:2d}, lambda = {lam:2d}: {STEPS + q.shape[0]} steps on {NX} x {NY} cells in {time.time() - t1:.1f} s", flush=True)
print(f"stored frames: {sum(len(r['frames']) for r in RUNS) / 1e6:.1f} MB")
print(f"all {len(RUNS)} runs in {time.time() - t0:.1f} s; frames stored at {NX // DOWN} x {NY // DOWN}")
DATA = dict(nx=NX // DOWN, ny=NY // DOWN, xwall=X_WALL // DOWN, slit=SLIT_W, seps=SEPARATIONS, lams=WAVELENGTHS, runs=RUNS)


In [ ]:
import json, html as _h
from IPython.display import HTML, display
PAGE = r'''<!doctype html><html><head><meta charset="utf-8">
<link href="https://fonts.googleapis.com/css2?family=Instrument+Serif&family=IBM+Plex+Sans:wght@400;500&family=IBM+Plex+Mono&display=swap" rel="stylesheet">
<style>
:root{--bg:#ece3d2;--panel:#f8f2e6;--paper:#fcf8f0;--ink:#23160f;--muted:#6f5b4c;--rule:#dccdb4;--wine:#7d1d2f;--wdeep:#5a0f21}
body{margin:0;background:var(--bg);color:var(--ink);font-family:"IBM Plex Sans",sans-serif}
.wrap{max-width:900px;margin:0 auto;padding:16px}
.eyebrow{font:500 11.5px "IBM Plex Mono",monospace;letter-spacing:.14em;text-transform:uppercase;color:var(--wine)}
h1{font:400 40px/1.1 "Instrument Serif",serif;color:var(--wdeep);margin:4px 0 6px}
.lede{color:var(--muted);margin:0 0 12px}
.card{background:var(--panel);border:1.5px solid #3f2c23;border-radius:12px;padding:14px;box-shadow:0 2px 8px rgba(0,0,0,.06)}
.fig{display:flex;gap:10px} canvas{background:var(--paper);border:1px solid var(--rule);border-radius:6px;max-width:100%}
.ctl{display:grid;grid-template-columns:150px 1fr 70px;gap:8px 12px;align-items:center;margin-top:12px;font-size:14px}
.ctl output,.read{font-family:"IBM Plex Mono",monospace;font-size:13px}
input[type=range]{accent-color:var(--wine);width:100%}
button{font:500 13px "IBM Plex Mono",monospace;background:var(--wine);color:#fff;border:0;border-radius:6px;padding:7px 14px;cursor:pointer}
button.ghost{background:transparent;color:var(--wine);border:1px solid var(--wine)}
.read{margin-top:10px;color:var(--muted)} .note{font-size:12.5px;color:var(--muted);margin-top:8px}
</style></head><body><div class="wrap">
<div class="eyebrow">Ripple tank · computed on Kaggle · played back</div>
<h1>A plane wave through two slits</h1>
<p class="lede">Every frame below was computed in the notebook (2D wave equation, finite differences). The page only plays the stored results; every slider position is its own computed run.</p>
<div class="card"><div class="fig"><canvas id="f" width="640" height="400"></canvas><canvas id="b" width="200" height="400"></canvas></div>
<div class="ctl">
<label>Slit separation d</label><input id="sd" type="range" min="0" max="__ND__" step="1" value="__MD__"><output id="od"></output>
<label>Wavelength λ</label><input id="sl" type="range" min="0" max="__NL__" step="1" value="__ML__"><output id="ol"></output></div>
<div style="margin-top:12px"><button id="go">Start</button> <button id="st" class="ghost">Step</button></div>
<div class="read" id="rd"></div>
<div class="note">Right panel: time-averaged intensity on the screen line (bars, computed), and the far-field two-slit formula (line). The screen is not far from the slits, so the formula is only a guide.</div>
</div></div>
<script>
const D=__DATA__;
const F=document.getElementById("f"),g=F.getContext("2d"),B=document.getElementById("b"),h=B.getContext("2d");
const img=g.createImageData(D.nx,D.ny),off=document.createElement("canvas");off.width=D.nx;off.height=D.ny;const og=off.getContext("2d");
const cache={};async function load(r){const key=r.d+"_"+r.lam;if(!cache[key]){const s=atob(r.frames),a=new Uint8Array(s.length);for(let i=0;i<s.length;i++)a[i]=s.charCodeAt(i);const st=new Blob([a]).stream().pipeThrough(new DecompressionStream("deflate"));cache[key]=new Uint8Array(await new Response(st).arrayBuffer())}return cache[key]}
function bytes(r){return cache[r.d+"_"+r.lam]}
let run=null,k=0,playing=false,last=0;
async function pick(){const d=D.seps[+sd.value],l=D.lams[+sl.value];const r=D.runs.find(r=>r.d==d&&r.lam==l);await load(r);run=r;k=0;od.textContent=d+" cells";ol.textContent=l+" cells";bars();draw()}
function col(v){const t=(v-128)/127;if(t>=0){return[252-t*(252-125),248-t*(248-29),240-t*(240-47)]}const s=-t;return[252-s*(252-51),248-s*(248-98),240-s*(240-127)]}
function draw(){const a=bytes(run),n=D.nx*D.ny,o=k*n;for(let j=0;j<D.ny;j++)for(let i=0;i<D.nx;i++){const v=a[o+i*D.ny+j],c=col(v),p=4*((D.ny-1-j)*D.nx+i);img.data[p]=c[0];img.data[p+1]=c[1];img.data[p+2]=c[2];img.data[p+3]=255}
og.putImageData(img,0,0);g.imageSmoothingEnabled=true;g.drawImage(off,0,0,F.width,F.height);
const sx=F.width/D.nx;g.fillStyle="#23160f";g.fillRect(D.xwall*sx,0,2*sx,F.height);
const sy=F.height/(2*D.ny);g.clearRect(D.xwall*sx,0,2*sx,0);for(const yc of [D.ny- run.d/2, D.ny+run.d/2]){g.clearRect(D.xwall*sx,F.height-(yc+D.slit/2)*sy,2*sx,D.slit*sy)}
rd.textContent="frame "+(k+1)+" of "+run.n+" (one period of the steady pattern) · run "+(D.runs.indexOf(run)+1)+" of "+D.runs.length+" · d = "+run.d+" · λ = "+run.lam+" · d/λ = "+(run.d/run.lam).toFixed(2)}
function bars(){h.clearRect(0,0,B.width,B.height);const n=run.I.length,dy=B.height/n;h.fillStyle="rgba(125,29,47,.55)";
for(let j=0;j<n;j++){h.fillRect(0,B.height-(j+1)*dy,run.I[j]*B.width*.9,dy+.5)}
h.beginPath();h.strokeStyle="#23160f";h.lineWidth=1.2;for(let j=0;j<n;j++){const X=run.fr[j]*B.width*.9,Y=B.height-(j+.5)*dy;j?h.lineTo(X,Y):h.moveTo(X,Y)}h.stroke();
h.fillStyle="#6f5b4c";h.font="11px IBM Plex Mono";h.fillText("intensity →",6,14)}
function loop(t){if(!playing)return;if(t-last>45){last=t;k=(k+1)%run.n;draw()}requestAnimationFrame(loop)}
go.onclick=()=>{playing=!playing;go.textContent=playing?"Pause":"Start";if(playing)requestAnimationFrame(loop)};
st.onclick=()=>{k=(k+1)%run.n;draw()};sd.oninput=pick;sl.oninput=pick;pick();
</script></body></html>'''
page = (PAGE.replace("__DATA__", json.dumps(DATA)).replace("__ND__", str(len(DATA["seps"]) - 1)).replace("__MD__", str(len(DATA["seps"]) // 2))
        .replace("__NL__", str(len(DATA["lams"]) - 1)).replace("__ML__", str(len(DATA["lams"]) // 2)))
out = "/kaggle/working/ripple_tank.html" if os.path.isdir("/kaggle/working") else "ripple_tank.html"
open(out, "w").write(page)
print(f"panel written to {out} ({len(page)/1e6:.1f} MB)")
display(HTML(f'<iframe srcdoc="{_h.escape(page)}" style="width:100%;height:760px;border:0"></iframe>'))
